<a href="https://colab.research.google.com/github/sajina464/C-Programs/blob/main/day_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

documents = [
    {
        "id": "D1",
        "text": (
            "Pay credit card bills and loan EMIs on time. "
            "Late payments can negatively affect credit scores. "
            "Check credit reports regularly for errors."
        )
    },
    {
        "id": "D2",
        "text": (
            "Keeping credit utilization low is generally helpful. "
            "Around 30 percent or less is a commonly used guideline. "
            "Lower utilization does not guarantee a higher score."
        )
    },
    {
        "id": "D3",
        "text": (
            "A debt-to-income ratio compares monthly debt payments "
            "with gross monthly income. Calculate it by dividing "
            "monthly debt payments by gross monthly income and "
            "multiplying by 100."
        )
    },
    {
        "id": "D4",
        "text": (
            "Loan approval depends on lender requirements, income, "
            "credit history, existing debts and other eligibility "
            "criteria. A credit score alone cannot guarantee approval."
        )
    },
    {
        "id": "D5",
        "text": (
            "Review credit reports for incorrect accounts or payment "
            "records. Dispute errors with the relevant credit bureau "
            "and provide supporting documents."
        )
    },
    {
        "id": "D6",
        "text": (
            "Build an emergency fund to cover unexpected expenses. "
            "Create a budget, track spending and prioritize essential "
            "payments before discretionary spending."
        )
    }
]

doc_df = pd.DataFrame(documents)
display(doc_df)
print("Total documents:", len(doc_df))


,id,text
0,D1,Pay credit card bills and loan EMIs on time. L...
1,D2,Keeping credit utilization low is generally he...
2,D3,A debt-to-income ratio compares monthly debt p...
3,D4,"Loan approval depends on lender requirements, ..."
4,D5,Review credit reports for incorrect accounts o...
5,D6,Build an emergency fund to cover unexpected ex...


Total documents: 6


In [3]:

test_cases = [
    {
        "query": "How can I improve my credit score?",
        "relevant_ids": ["D1", "D2", "D5"],
        "expected_keywords": ["credit", "payments", "score"]
    },
    {
        "query": "How do I calculate my debt to income ratio?",
        "relevant_ids": ["D3"],
        "expected_keywords": ["monthly debt", "monthly income", "100"]
    },
    {
        "query": "Will a good credit score guarantee a loan?",
        "relevant_ids": ["D4"],
        "expected_keywords": ["loan approval", "cannot guarantee"]
    },
    {
        "query": "How can I reduce my credit card utilization?",
        "relevant_ids": ["D2"],
        "expected_keywords": ["utilization", "30 percent"]
    },
    {
        "query": "What should I do about errors in my credit report?",
        "relevant_ids": ["D5", "D1"],
        "expected_keywords": ["credit report", "errors", "dispute"]
    },
    {
        "query": "How can I manage unexpected expenses?",
        "relevant_ids": ["D6"],
        "expected_keywords": ["emergency fund", "budget"]
    }
]

print("Total evaluation queries:", len(test_cases))


Total evaluation queries: 6


In [4]:

corpus = [doc["text"] for doc in documents]

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

document_vectors = vectorizer.fit_transform(corpus)

def retrieve_documents(query, top_k=3):
    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector, document_vectors
    )[0]

    ranked_indices = similarities.argsort()[::-1][:top_k]

    results = []
    for idx in ranked_indices:
        results.append({
            "id": documents[idx]["id"],
            "text": documents[idx]["text"],
            "similarity": float(similarities[idx])
        })

    return results

sample = retrieve_documents(
    "How can I improve my credit score?"
)

for item in sample:
    print(item["id"], round(item["similarity"], 3))
    print(item["text"], "\n")


D4 0.3
Loan approval depends on lender requirements, income, credit history, existing debts and other eligibility criteria. A credit score alone cannot guarantee approval. 

D1 0.129
Pay credit card bills and loan EMIs on time. Late payments can negatively affect credit scores. Check credit reports regularly for errors. 

D2 0.124
Keeping credit utilization low is generally helpful. Around 30 percent or less is a commonly used guideline. Lower utilization does not guarantee a higher score. 



In [5]:

def normalize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def generate_answer(query, retrieved_docs):
    """
    Extractive baseline: returns the top retrieved document.
    Replace this function with your actual RAG/LLM answer function
    when integrating your existing assistant.
    """
    if not retrieved_docs or retrieved_docs[0]["similarity"] == 0:
        return "I could not find sufficient information in the knowledge base."

    return retrieved_docs[0]["text"]


def evaluate_all(k=3):
    evaluation_rows = []

    for case in test_cases:
        query = case["query"]
        relevant_ids = set(case["relevant_ids"])

        retrieved = retrieve_documents(query, top_k=k)
        retrieved_ids = [d["id"] for d in retrieved]

        # Retrieval metrics
        relevant_retrieved = sum(
            1 for doc_id in retrieved_ids
            if doc_id in relevant_ids
        )

        precision = relevant_retrieved / k
        recall = relevant_retrieved / len(relevant_ids)

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        reciprocal_rank = 0
        for rank, doc_id in enumerate(retrieved_ids, start=1):
            if doc_id in relevant_ids:
                reciprocal_rank = 1 / rank
                break

        # Response evaluation
        answer = generate_answer(query, retrieved)
        answer_tokens = set(normalize(answer))
        expected = case["expected_keywords"]

        keyword_coverage = sum(
            1 for keyword in expected
            if keyword.lower() in answer.lower()
        ) / len(expected)

        context = " ".join(d["text"] for d in retrieved)
        context_tokens = set(normalize(context))

        answer_content = set(normalize(answer))
        answer_content = {
            token for token in answer_content
            if token not in {
                "the", "a", "an", "and", "or", "is",
                "are", "to", "of", "in", "on", "with"
            }
        }

        groundedness = (
            len(answer_content & context_tokens) /
            len(answer_content)
            if answer_content else 0
        )

        evaluation_rows.append({
            "query": query,
            "retrieved_ids": ", ".join(retrieved_ids),
            "precision_at_k": round(precision, 3),
            "recall_at_k": round(recall, 3),
            "retrieval_f1": round(f1, 3),
            "mrr": round(reciprocal_rank, 3),
            "keyword_coverage": round(keyword_coverage, 3),
            "groundedness_proxy": round(groundedness, 3),
            "answer": answer
        })

    return pd.DataFrame(evaluation_rows)


results_df = evaluate_all(k=3)

print("RETRIEVAL AND RESPONSE EVALUATION")
display(results_df)


RETRIEVAL AND RESPONSE EVALUATION


,query,retrieved_ids,precision_at_k,recall_at_k,retrieval_f1,mrr,keyword_coverage,groundedness_proxy,answer
0,How can I improve my credit score?,"D4, D1, D2",0.667,0.667,0.667,0.5,0.667,1.0,"Loan approval depends on lender requirements, ..."
1,How do I calculate my debt to income ratio?,"D3, D4, D5",0.333,1.000,0.500,1.0,1.000,1.0,A debt-to-income ratio compares monthly debt p...
2,Will a good credit score guarantee a loan?,"D4, D1, D2",0.333,1.000,0.500,1.0,1.000,1.0,"Loan approval depends on lender requirements, ..."
3,How can I reduce my credit card utilization?,"D1, D2, D5",0.333,1.000,0.500,0.5,0.000,1.0,Pay credit card bills and loan EMIs on time. L...
4,What should I do about errors in my credit rep...,"D1, D5, D4",0.667,1.000,0.800,1.0,0.667,1.0,Pay credit card bills and loan EMIs on time. L...
5,How can I manage unexpected expenses?,"D6, D5, D4",0.333,1.000,0.500,1.0,1.000,1.0,Build an emergency fund to cover unexpected ex...


In [6]:

metric_columns = [
    "precision_at_k",
    "recall_at_k",
    "retrieval_f1",
    "mrr",
    "keyword_coverage",
    "groundedness_proxy"
]

summary = results_df[metric_columns].mean().round(3)

print("OVERALL EVALUATION SCORES")
display(summary.to_frame(name="Average Score"))

results_df.to_csv(
    "financial_rag_quality_report.csv",
    index=False
)

summary.to_csv(
    "financial_rag_quality_summary.csv",
    header=["Average Score"]
)

print("Evaluation completed!")
print("Detailed report: financial_rag_quality_report.csv")
print("Summary report: financial_rag_quality_summary.csv")


OVERALL EVALUATION SCORES


,Average Score
precision_at_k,0.444
recall_at_k,0.944
retrieval_f1,0.578
mrr,0.833
keyword_coverage,0.722
groundedness_proxy,1.000


Evaluation completed!
Detailed report: financial_rag_quality_report.csv
Summary report: financial_rag_quality_summary.csv


In [7]:

from google.colab import files

files.download("financial_rag_quality_report.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>